In [74]:
from dotenv import load_dotenv
import os 

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [75]:
!pip install -q langchain-openai langchain-core  

In [76]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage , ToolMessage
import requests

In [77]:
# tool create 
@tool 
def multiply(a:int , b:int) -> int:
    """Given 2 numbers a and b this tool returns their product"""
    return a * b 

In [78]:
print(multiply.invoke({'a':3 , 'b':5}))

15


In [79]:
multiply.name

'multiply'

In [80]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [81]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

Tools Binding

In [82]:
llm = HuggingFaceEndpoint(
    repo_id = "meta-llama/Llama-3.3-70B-Instruct" ,
    task = "text-generation"
)

model = ChatHuggingFace(llm = llm)

In [83]:
llm_with_tools = model.bind_tools([multiply])

In [84]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="I'm doing well, thanks for asking. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 234, 'total_tokens': 259}, 'model_name': 'meta-llama/Llama-3.3-70B-Instruct', 'system_fingerprint': 'fp_f8b414701e', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019edfca-83dd-7630-91fa-3a1afb268c84-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 234, 'output_tokens': 25, 'total_tokens': 259})

In [85]:
query = HumanMessage('can you mutliply 3 with 10')

In [86]:
messages = [query]

In [87]:
messages

[HumanMessage(content='can you mutliply 3 with 10', additional_kwargs={}, response_metadata={})]

In [88]:
ai_msg = llm_with_tools.invoke(messages)

In [89]:
messages.append(ai_msg)

In [90]:
print(messages)

[HumanMessage(content='can you mutliply 3 with 10', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":3,"b":10}', 'name': 'multiply', 'description': None}, 'id': 'w9ehm2pzm', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 240, 'total_tokens': 259}, 'model_name': 'meta-llama/Llama-3.3-70B-Instruct', 'system_fingerprint': 'fp_43d97c5965', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019edfcb-ab51-7520-aa94-f9de9736fc83-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'w9ehm2pzm', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 240, 'output_tokens': 19, 'total_tokens': 259})]


In [91]:
tool_result = multiply.invoke(result.tool_calls[0])

In [92]:
tool_result

ToolMessage(content='30', name='multiply', tool_call_id='dt6fg1m94')

In [93]:
messages.append(tool_result)

In [94]:
messages

[HumanMessage(content='can you mutliply 3 with 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":3,"b":10}', 'name': 'multiply', 'description': None}, 'id': 'w9ehm2pzm', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 240, 'total_tokens': 259}, 'model_name': 'meta-llama/Llama-3.3-70B-Instruct', 'system_fingerprint': 'fp_43d97c5965', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019edfcb-ab51-7520-aa94-f9de9736fc83-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'w9ehm2pzm', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 240, 'output_tokens': 19, 'total_tokens': 259}),
 ToolMessage(content='30', name='multiply', tool_call_id='dt6fg1m94')]

In [95]:
llm_with_tools.invoke(messages).content

'The result of multiplying 3 by 10 is 30.'